## 11.2.1 蒐集資料：Kaggle貓與狗資料集

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
folder="/content/drive/MyDrive/資料科學自學聖經/ch11"

下載貓狗資料集

修正後網址:https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

In [1]:
!wget --no-check-certificate "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

--2024-07-26 06:41:33--  https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.220.113.200, 2a02:26f0:6d00:3b6::317f, 2a02:26f0:6d00:39f::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.220.113.200|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘kagglecatsanddogs_5340.zip’

kagglecatsanddogs_5 100%[===================>] 786.67M   141MB/s    in 6.7s    

2024-07-26 06:41:40 (117 MB/s) - ‘kagglecatsanddogs_5340.zip’ saved [824887076/824887076]



解壓縮

In [2]:
!unzip "kagglecatsanddogs_5340.zip"

串流輸出內容已截斷至最後 5000 行。
  inflating: PetImages/Dog/5500.jpg  
  inflating: PetImages/Dog/5501.jpg  
  inflating: PetImages/Dog/5502.jpg  
  inflating: PetImages/Dog/5503.jpg  
  inflating: PetImages/Dog/5504.jpg  
  inflating: PetImages/Dog/5505.jpg  
  inflating: PetImages/Dog/5506.jpg  
  inflating: PetImages/Dog/5507.jpg  
  inflating: PetImages/Dog/5508.jpg  
  inflating: PetImages/Dog/5509.jpg  
  inflating: PetImages/Dog/551.jpg   
  inflating: PetImages/Dog/5510.jpg  
  inflating: PetImages/Dog/5511.jpg  
  inflating: PetImages/Dog/5512.jpg  
  inflating: PetImages/Dog/5513.jpg  
  inflating: PetImages/Dog/5514.jpg  
  inflating: PetImages/Dog/5515.jpg  
  inflating: PetImages/Dog/5516.jpg  
  inflating: PetImages/Dog/5517.jpg  
  inflating: PetImages/Dog/5518.jpg  
  inflating: PetImages/Dog/5519.jpg  
  inflating: PetImages/Dog/552.jpg   
  inflating: PetImages/Dog/5520.jpg  
  inflating: PetImages/Dog/5521.jpg  
  inflating: PetImages/Dog/5522.jpg  
  inflating: PetImages/Dog/55

讀取貓狗資料集

In [3]:
import tensorflow as tf
import os, cv2, glob

images=[]
labels=[]
dict_labels = {"Cat":0, "Dog":1}
for folders in glob.glob("/content/PetImages/*"):
    print(folders,"圖片讀取中…")
    for filename in os.listdir(folders):
        label=folders.split("/")[-1]
        try:
            img=cv2.imread(os.path.join(folders,filename))
            img = cv2.resize(img,dsize=(80,80))
            if img is not None:
                images.append(img)
                labels.append(dict_labels[label])
        except:
            pass
print("圖片讀取完畢!")

/content/PetImages/Cat 圖片讀取中…
/content/PetImages/Dog 圖片讀取中…
圖片讀取完畢!


查看圖片和標籤的筆數

In [4]:
print('圖片數量：{}'.format(len(images)))
print('標籤數量：{}'.format(len(labels)))

圖片數量：24946
標籤數量：24946


## 11.2.2 資料預處理

資料集資料分割

In [5]:
from sklearn.model_selection import train_test_split
import numpy as np
#from keras.utils import np_utils
from tensorflow.python.keras.utils import np_utils #新版語法

train_feature,test_feature,train_label,test_label = \
train_test_split(images, labels, test_size=0.2)

train_feature=np.array(train_feature)
test_feature=np.array(test_feature)
train_label=np.array(train_label)
test_label=np.array(test_label)

顯示訓練和測試資料內容

In [6]:
print('訓練資料維度：{}'.format(train_feature.shape))
print('訓練標籤維度：{}'.format(train_label.shape))

訓練資料維度：(19956, 80, 80, 3)
訓練標籤維度：(19956,)


In [7]:
print('測試資料維度：{}'.format(test_feature.shape))
print('測試標籤維度：{}'.format(test_label.shape))

測試資料維度：(4990, 80, 80, 3)
測試標籤維度：(4990,)


圖片資料標準化

將圖片資料標準化為0-1之間的浮點數,以提高預測準確度。

In [8]:
train_feature = train_feature/255
test_feature = test_feature/255

標籤轉換為One-Hot編碼

In [9]:
train_label = np_utils.to_categorical(train_label)
test_label = np_utils.to_categorical(test_label)

## 11.2.3 建立卷積神經網路模型

載入模組與建立物件

In [10]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Dropout, Flatten, Dense

model = Sequential()

建立第一層卷積層

In [11]:
model.add(Conv2D(filters=8, kernel_size=(5,5), padding='same',
                 input_shape=(80, 80, 3), activation='relu'))

建立第一層池化層

In [12]:
model.add(MaxPooling2D(pool_size=(2, 2)))

建立第一層拋棄層

In [13]:
model.add(Dropout(0.2))

建立第二層卷積層、池化層與拋棄層

In [14]:
model.add(Conv2D(filters=16, kernel_size=(5,5),
                 padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

建立第三層卷積層、池化層與拋棄層

In [15]:
model.add(Conv2D(filters=32, kernel_size=(5,5),
                 padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

建立平坦層與拋棄層

In [16]:
model.add(Flatten())
model.add(Dropout(0.2))

建立全連結隱藏層

In [17]:
model.add(Dense(units=128, activation='relu'))

建立輸出層

In [18]:
model.add(Dense(units=2,activation='softmax'))

查看權重


In [19]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 80, 80, 8)         608       
                                                                 
 max_pooling2d (MaxPooling2  (None, 40, 40, 8)         0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 40, 40, 8)         0         
                                                                 
 conv2d_1 (Conv2D)           (None, 40, 40, 16)        3216      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 20, 20, 16)        0         
 g2D)                                                            
                                                                 
 dropout_1 (Dropout)         (None, 20, 20, 16)        0

## 11.2.4 訓練模型及評估準確率

訓練模型

In [20]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])
model.fit(x=train_feature, y=train_label, validation_split=0.2,
          epochs=20, batch_size=200, verbose=2)

Epoch 1/20
80/80 - 12s - loss: 0.6820 - accuracy: 0.5594 - val_loss: 0.6685 - val_accuracy: 0.5844 - 12s/epoch - 152ms/step
Epoch 2/20
80/80 - 2s - loss: 0.6189 - accuracy: 0.6621 - val_loss: 0.5927 - val_accuracy: 0.6979 - 2s/epoch - 29ms/step
Epoch 3/20
80/80 - 2s - loss: 0.5723 - accuracy: 0.7061 - val_loss: 0.5793 - val_accuracy: 0.6939 - 2s/epoch - 28ms/step
Epoch 4/20
80/80 - 2s - loss: 0.5375 - accuracy: 0.7278 - val_loss: 0.5339 - val_accuracy: 0.7325 - 2s/epoch - 30ms/step
Epoch 5/20
80/80 - 3s - loss: 0.5108 - accuracy: 0.7495 - val_loss: 0.4967 - val_accuracy: 0.7548 - 3s/epoch - 37ms/step
Epoch 6/20
80/80 - 3s - loss: 0.4922 - accuracy: 0.7578 - val_loss: 0.4815 - val_accuracy: 0.7683 - 3s/epoch - 39ms/step
Epoch 7/20
80/80 - 3s - loss: 0.4730 - accuracy: 0.7735 - val_loss: 0.4698 - val_accuracy: 0.7776 - 3s/epoch - 32ms/step
Epoch 8/20
80/80 - 2s - loss: 0.4545 - accuracy: 0.7834 - val_loss: 0.4718 - val_accuracy: 0.7698 - 2s/epoch - 28ms/step
Epoch 9/20
80/80 - 2s - loss:

評估準確率

In [21]:
scores = model.evaluate(test_feature, test_label)
print('\n準確率=',scores[1])

156/156 [==============================] - 1s 5ms/step - loss: 0.3947 - accuracy: 0.8178

準確率= 0.8178356885910034


## 11.2.5 完整貓狗辨識模型程式碼

In [22]:
# 先下載貓狗圖片
!wget --no-check-certificate "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
!unzip "kagglecatsanddogs_5340.zip"

串流輸出內容已截斷至最後 5000 行。
  inflating: PetImages/Dog/5500.jpg  
  inflating: PetImages/Dog/5501.jpg  
  inflating: PetImages/Dog/5502.jpg  
  inflating: PetImages/Dog/5503.jpg  
  inflating: PetImages/Dog/5504.jpg  
  inflating: PetImages/Dog/5505.jpg  
  inflating: PetImages/Dog/5506.jpg  
  inflating: PetImages/Dog/5507.jpg  
  inflating: PetImages/Dog/5508.jpg  
  inflating: PetImages/Dog/5509.jpg  
  inflating: PetImages/Dog/551.jpg   
  inflating: PetImages/Dog/5510.jpg  
  inflating: PetImages/Dog/5511.jpg  
  inflating: PetImages/Dog/5512.jpg  
  inflating: PetImages/Dog/5513.jpg  
  inflating: PetImages/Dog/5514.jpg  
  inflating: PetImages/Dog/5515.jpg  
  inflating: PetImages/Dog/5516.jpg  
  inflating: PetImages/Dog/5517.jpg  
  inflating: PetImages/Dog/5518.jpg  
  inflating: PetImages/Dog/5519.jpg  
  inflating: PetImages/Dog/552.jpg   
  inflating: PetImages/Dog/5520.jpg  
  inflating: PetImages/Dog/5521.jpg  
  inflating: PetImages/Dog/5522.jpg  
  inflating: PetImages/Dog/55

In [4]:
import os, cv2, glob
from sklearn.model_selection import train_test_split
import numpy as np
#from keras.utils import np_utils
from tensorflow.python.keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Dropout, Flatten, Dense
images=[]
labels=[]
dict_labels = {"Cat":0, "Dog":1}
for folders in glob.glob("/content/PetImages/*"):
    print(folders,"圖片讀取中…")
    for filename in os.listdir(folders):
        label=folders.split("/")[-1]
        try:
            img=cv2.imread(os.path.join(folders,filename))
            img = cv2.resize(img,dsize=(80,80))
            if img is not None:
                images.append(img)
                labels.append(dict_labels[label])
        except:
            pass
    print(folders,"圖片讀取完畢!")
train_feature,test_feature,train_label,test_label = \
train_test_split(images,labels,test_size=0.2)
train_feature=np.array(train_feature)
test_feature=np.array(test_feature)
train_label=np.array(train_label)
test_label=np.array(test_label)
train_feature = train_feature/255
test_feature = test_feature/255
train_label = np_utils.to_categorical(train_label)
test_label = np_utils.to_categorical(test_label)

model = Sequential()
model.add(Conv2D(filters=8, kernel_size=(5,5), padding='same',
                 input_shape=(80, 80, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))
model.add(Conv2D(filters=16, kernel_size=(5,5),
                 padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))
model.add(Conv2D(filters=32, kernel_size=(5,5),
                 padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dropout(0.2))
model.add(Dense(units=128, activation='relu'))
model.add(Dense(units=2,activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])
model.fit(x=train_feature, y=train_label,
          validation_split=0.2, epochs=20,
          batch_size=200, verbose=2)
scores = model.evaluate(test_feature, test_label)
print('\n準確率=',scores[1])
model.save('catdog_model.h5')

/content/PetImages/Cat 圖片讀取中…
/content/PetImages/Cat 圖片讀取完畢!
/content/PetImages/Dog 圖片讀取中…
/content/PetImages/Dog 圖片讀取完畢!
Epoch 1/20
80/80 - 12s - loss: 0.6883 - accuracy: 0.5427 - val_loss: 0.6508 - val_accuracy: 0.6453 - 12s/epoch - 153ms/step
Epoch 2/20
80/80 - 3s - loss: 0.6254 - accuracy: 0.6564 - val_loss: 0.5945 - val_accuracy: 0.6874 - 3s/epoch - 34ms/step
Epoch 3/20
80/80 - 2s - loss: 0.5809 - accuracy: 0.7036 - val_loss: 0.5588 - val_accuracy: 0.7179 - 2s/epoch - 30ms/step
Epoch 4/20
80/80 - 3s - loss: 0.5454 - accuracy: 0.7283 - val_loss: 0.5839 - val_accuracy: 0.6911 - 3s/epoch - 32ms/step
Epoch 5/20
80/80 - 3s - loss: 0.5176 - accuracy: 0.7401 - val_loss: 0.4878 - val_accuracy: 0.7705 - 3s/epoch - 42ms/step
Epoch 6/20
80/80 - 3s - loss: 0.4842 - accuracy: 0.7685 - val_loss: 0.4903 - val_accuracy: 0.7605 - 3s/epoch - 35ms/step
Epoch 7/20
80/80 - 3s - loss: 0.4656 - accuracy: 0.7791 - val_loss: 0.4655 - val_accuracy: 0.7823 - 3s/epoch - 32ms/step
Epoch 8/20
80/80 - 2s - loss

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## 11.2.6 預測自己的貓狗圖片

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from keras.models import load_model
import glob,cv2

def show_images_labels_predictions(images, labels,
                  predictions,start_id, num=10):
    plt.figure(figsize=(12, 14))
    if num>25: num=25
    for i in range(0, num):
        ax=plt.subplot(5,5, 1+i)
        ax.imshow(images[start_id])
        if( len(predictions) > 0 ) :
            title = 'ai = ' + str(predictions[start_id])
            title += (' (o)' if predictions[start_id]== \
                      labels[start_id] else ' (x)')
            title += '\nlabel = ' + str(labels[start_id])
        else :
            title = 'label = ' + str(labels[start_id])
        ax.set_title(title,fontsize=12)
        ax.set_xticks([]);ax.set_yticks([])
        start_id+=1
    plt.show()


files = glob.glob("/content/*.jpg" )
#files = glob.glob("/content/drive/MyDrive/*.jpg" )
test_feature=[]
test_label=[]
dict_labels = {"Cat":0, "Dog":1}

for file in files:
    img=cv2.imread(file)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, dsize=(80,80))
    test_feature.append(img)
    label=file[10:13]
    test_label.append(dict_labels[label])
    #test_feature = np.array(test_feature)
    test_feature = np.array(test_feature).reshape(len(test_feature),40,40,3).astype('float32')
    test_label = np.array(test_label)

test_feature_vector=test_feature
test_feature_n = test_feature_vector / 255 #有錯!!

try:

  model = load_model('/content/catdog_model.h5')
  #prediction = tf.config.run_functions_eagerly(True)
  prediction = model.predict(test_feature_vector)
  prediction = np.argmax(prediction,axis=1)
  show_images_labels_predictions(test_feature,test_label,prediction,0,len(test_feature))
except:
  print("模型未建立!")

TypeError: unsupported operand type(s) for /: 'list' and 'int'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras.models import load_model
import glob,cv2

def show_images_labels_predictions(images, labels,
                  predictions,start_id, num=10):
    plt.figure(figsize=(12, 14))
    if num>25: num=25
    for i in range(0, num):
        ax=plt.subplot(5,5, 1+i)
        ax.imshow(images[start_id])
        if( len(predictions) > 0 ) :
            title = 'ai = ' + str(predictions[start_id])
            title += (' (o)' if predictions[start_id]== \
                      labels[start_id] else ' (x)')
            title += '\nlabel = ' + str(labels[start_id])
        else :
            title = 'label = ' + str(labels[start_id])
        ax.set_title(title,fontsize=12)
        ax.set_xticks([]);ax.set_yticks([])
        start_id+=1
    plt.show()


files = glob.glob("/content/*.jpg" )
#files = glob.glob("/content/drive/MyDrive/*.jpg" )
test_feature=[]
test_label=[]
dict_labels = {"Cat":0, "Dog":1}

for file in files:
    img=cv2.imread(file)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, dsize=(80,80))
    test_feature.append(img)
    label=file[10:13]
    test_label.append(dict_labels[label])
    test_feature=np.array(test_feature)
    test_label=np.array(test_label)

test_feature_vector = test_feature.reshape(len(test_feature),80,80,3).astype('float32')

test_feature_n = test_feature_vector/255

try:
  model = load_model('/content/catdog_model.h5')
  #prediction = tf.config.run_functions_eagerly(True)
  prediction = model.predict(test_feature_n)
  prediction = np.argmax(prediction,axis=1)
  show_images_labels_predictions(test_feature,test_label,prediction,0,len(test_feature))
except:
  print("模型未建立!")

AttributeError: ignored

## 11.2.7 Gradio展示貓狗圖片辨識

In [2]:
!pip install --upgrade gradio

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.2/318.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━

In [3]:
from tensorflow.keras.models import load_model
import gradio as gr


model = load_model("/content/catdog_model.h5")

def catdog(image):
    image = image.reshape(1, 80, 80, 3)
    prediction = model.predict(image).tolist()[0]
    class_names = ["Cat", "Dog"]
    return {class_names[i]: prediction[i] for i in range(2)}

inp = gr.inputs.Image(shape=(80, 80), source="upload")
out = gr.outputs.Label(num_top_classes=2, label='預測結果')
grobj = gr.Interface(fn=catdog, inputs=inp,
                     outputs=out, title="貓狗圖片辨識")
grobj.launch()

OSError: No file or directory found at /content/catdog_model.h5